# Nearest-Facility and Closest-Facility Allocation by Bicycle — Triangeln, Malmö

**Study area:** 1 km × 1 km square centred on Triangeln station, Malmö.
**Mode:** cycling. **Facilities:** schools and hospitals.

## Question

For every home inside the study square: *which school and which hospital is nearest by
bike, how long does that ride take, and how does the area divide up into catchments?*

Two classical network-analysis operations answer that, and this notebook runs both:

| Operation | Question it answers | Output here |
|---|---|---|
| **Nearest facility** | For each demand point, which facility is closest and at what cost? | A ride time and a facility label per building |
| **Closest-facility allocation** | Partitioning the network by its nearest facility | Catchments, per-facility load, approach corridors |

## Why cycling changes the answer

A straight-line or car-network buffer gets this wrong in Malmö. Cycling follows a network
with its own topology — cycleways cut through blocks a car cannot use, and one-ways
constrain a rider differently. It also has a *comfort* dimension: 1 km on a protected
cycleway and 1 km in mixed traffic are the same distance and not the same trip. So every
edge carries two impedances:

- `t_min` — real ride time at a fixed cycling speed.
- `t_perceived` — that time multiplied by a traffic-stress factor, so a rider who avoids
  mixed traffic is modelled as taking the route they would actually choose.

The allocation is run under both, and §11 measures how many homes change their nearest
school when comfort is priced in.

## Method

1. Build the AOI square, plus a routing buffer so rides can leave and re-enter the frame.
2. Download the OSM bicycle network over the buffered box; classify every edge as
   protected / painted lane / mixed traffic; attach both impedances.
3. Collect school and hospital features over the buffered box, resolve them to **sites**
   (departments inside one campus are not separate hospitals), and snap each to the network.
4. Build demand points from residential buildings inside the AOI, weighted by floor area.
5. **Nearest facility** — one multi-source Dijkstra per facility type, run on the *reversed*
   graph from all facilities at once, so each node learns its nearest facility and cost in
   a single pass.
6. **Closest-facility allocation** — label every node, cut the network into catchments,
   load the demand onto its route, and measure per-facility load and coverage.

Outputs (maps, tables, GeoPackage) are written to `../OUT/`.

## 0. Dependencies

In [ ]:
# Colab / fresh environment. Skip if your env already has these.
# !pip install -q osmnx>=1.9 geopandas networkx matplotlib mapclassify pyogrio

In [ ]:
import heapq
import math
import socket
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely.geometry import LineString, Point
from shapely.ops import unary_union

print("osmnx     ", ox.__version__)
print("networkx  ", nx.__version__)
print("geopandas ", gpd.__version__)

### Version-compatibility shims

OSMnx moved several functions between submodules in the 1.x → 2.x transition. These shims
resolve whichever name the installed version exposes, so the notebook runs on both. They
match the shims in the companion fragmentation notebook, with `features_from_polygon`
added for the facility and building queries.

In [ ]:
def _resolve(name, *modules):
    for mod in modules:
        if mod is None:
            continue
        fn = getattr(mod, name, None)
        if callable(fn):
            return fn
    return None


def graph_to_gdfs(G, **kwargs):
    fn = _resolve(
        "graph_to_gdfs",
        getattr(ox, "convert", None),      # osmnx >= 2.0
        getattr(ox, "utils_graph", None),  # osmnx 1.x
        ox,                                 # osmnx 1.x top-level alias
    )
    if fn is None:
        raise ImportError("Could not locate osmnx graph_to_gdfs")
    return fn(G, **kwargs)


def graph_from_polygon(polygon, **kwargs):
    fn = _resolve("graph_from_polygon", getattr(ox, "graph", None), ox)
    if fn is None:
        raise ImportError("Could not locate osmnx graph_from_polygon")
    return fn(polygon, **kwargs)


def project_graph(G, **kwargs):
    fn = _resolve("project_graph", getattr(ox, "projection", None), ox)
    if fn is None:
        raise ImportError("Could not locate osmnx project_graph")
    return fn(G, **kwargs)


def features_from_polygon(polygon, tags):
    fn = _resolve("features_from_polygon", getattr(ox, "features", None), ox)
    if fn is None:  # osmnx 1.x called it geometries_from_polygon
        fn = _resolve("geometries_from_polygon", getattr(ox, "geometries", None), ox)
    if fn is None:
        raise ImportError("Could not locate osmnx features_from_polygon")
    return fn(polygon, tags)


def _which(name, *modules):
    for mod in modules:
        if mod is not None and callable(getattr(mod, name, None)):
            return f"{getattr(mod, '__name__', mod)}.{name}"
    return "UNRESOLVED"


print("graph_to_gdfs        ->", _which("graph_to_gdfs", getattr(ox, "convert", None),
                                        getattr(ox, "utils_graph", None), ox))
print("graph_from_polygon   ->", _which("graph_from_polygon", getattr(ox, "graph", None), ox))
print("project_graph        ->", _which("project_graph", getattr(ox, "projection", None), ox))
print("features_from_polygon->", _which("features_from_polygon", getattr(ox, "features", None),
                                        getattr(ox, "geometries", None), ox))

## 1. Configuration

In [ ]:
# --- Study area -------------------------------------------------------------
# Triangeln station, Malmö. Same centre as the companion fragmentation notebook,
# so the two analyses describe exactly the same square.
CENTER_LAT, CENTER_LON = 55.5928, 13.0000

AOI_SIDE_M = 1000.0          # square side -> 1000 m x 1000 m = 1.00 km^2
CRS_METRIC = "EPSG:3006"     # SWEREF99 TM - the national projected CRS for Sweden
CRS_WGS84 = "EPSG:4326"

# A 1 km box is smaller than a cycling trip. Two buffers keep the edge honest:
#   - facilities are searched in a wider box, because a rider at the AOI edge will
#     obviously use a school 300 m outside the frame;
#   - the routing graph is wider still, so the ride to those facilities is routed on
#     real street, not clipped at the boundary.
FACILITY_SEARCH_BUFFER_M = 1000.0
ROUTING_BUFFER_M = 1500.0

# --- Facility definitions ---------------------------------------------------
# Each tier is (OSM tag filter, display label). Hospitals are strict: amenity=hospital
# or healthcare=hospital. To fold in Swedish primary care (vardcentral), add
# "clinic"/"doctors" to the amenity list - see the note in section 5.
FACILITY_SPECS = {
    "school":   {"tags": {"amenity": ["school"]},
                 "label": "School",   "plural": "schools"},
    "hospital": {"tags": {"amenity": ["hospital"], "healthcare": ["hospital"]},
                 "label": "Hospital", "plural": "hospitals"},
}

# Departments mapped as separate nodes inside one campus are not separate facilities.
# A point feature this close to a site polygon is a sub-feature of it and is absorbed.
SITE_ABSORB_M = 25.0
# Two standalone point features this close are the same site mapped twice. Polygons are
# never merged on distance - adjacent schools share fences.
SITE_DEDUPE_M = 60.0

# --- Cycling impedance ------------------------------------------------------
CYCLING_SPEED_KMH = 15.0     # door-to-door urban average, stops included

# Traffic-stress multipliers on time. A minute in mixed traffic "costs" a rider more
# than a minute on a protected cycleway; these are the standard ordering, calibrated
# loosely to LTS route-choice studies. Set USE_COMFORT_WEIGHTS = False to route on
# plain time only.
STRESS_FACTOR = {"protected": 1.00, "lane": 1.15, "mixed": 1.45}
USE_COMFORT_WEIGHTS = True

# Combined foot/cycle paths (gang- och cykelvag) count as protected: they are separated
# from motor traffic, which is what the stress factor prices.
INCLUDE_SHARED_PATHS = True

# --- Demand model -----------------------------------------------------------
# Residential buildings inside the AOI, weighted by a floor-area proxy.
DEMAND_WEIGHT = "floor_area"          # "floor_area" | "footprint" | "equal"
DEFAULT_LEVELS = 4.0                  # central Malmo perimeter block, where untagged
MIN_BUILDING_AREA_M2 = 40.0           # drops sheds, garages, bin stores
NON_RESIDENTIAL = {
    "hospital", "school", "university", "college", "kindergarten", "retail",
    "commercial", "office", "industrial", "warehouse", "garage", "garages",
    "carport", "roof", "shed", "hut", "parking", "church", "chapel", "cathedral",
    "mosque", "synagogue", "train_station", "transportation", "sports_centre",
    "sports_hall", "stadium", "grandstand", "service", "government", "public",
    "hotel", "civic", "fire_station", "greenhouse", "bunker", "toilets", "kiosk",
}

# --- Reporting --------------------------------------------------------------
TIME_BANDS_MIN = [2, 4, 6, 8, 10]     # isochrone band edges, minutes
COVERAGE_TARGET_MIN = {"school": 5.0, "hospital": 10.0}
SNAP_WARN_M = 150.0                   # flag demand points this far from the network
CATCHMENT_BAND_M = 45.0               # half-width of the catchment ribbon drawn on the map

# --- Paths ------------------------------------------------------------------
OUT_DIR = Path("../OUT")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- OSMnx settings ---------------------------------------------------------
ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.requests_timeout = 300


# --- Choose a working Overpass endpoint -------------------------------------
# Overpass is a free, shared, volunteer-funded service and this notebook is a demanding
# client. Three things go wrong in practice, and all three surface as one opaque failure:
#
# 1. One name, several machines. overpass-api.de is a handful of servers behind a single
#    hostname and they do not fail together - one can refuse TCP while another serves.
# 2. OSMnx pins one of them for the whole session *on purpose*: it wants the rate-limit
#    check and the query itself to reach the same machine, or it would violate the other
#    machine's slot timing. It picks that IP with socket.gethostbyname, which returns
#    whichever A record comes first. If that one is down, every request in the session
#    fails identically and retrying cannot help.
# 3. A server that accepts a connection may still refuse the work - 429 when you are over
#    quota, 502/504 when it is overloaded. A TCP handshake proves nothing.
#
# So: probe each candidate with a real (tiny) query, use the first that actually answers,
# and pin that backend for the session. OSMnx keeps its one-server guarantee, on a server
# that is up and willing.
OVERPASS_CANDIDATES = [
    "https://overpass-api.de/api",           # the main instance
    "https://overpass.kumi.systems/api",     # mirrors, in order of preference
    "https://overpass.private.coffee/api",
]

_ORIGINAL_GETHOSTBYNAME = socket.gethostbyname
_ORIGINAL_GETADDRINFO = socket.getaddrinfo
_PINNED_HOSTS = {}


def _pinned_gethostbyname(host):
    return _PINNED_HOSTS.get(host) or _ORIGINAL_GETHOSTBYNAME(host)


def _pinned_getaddrinfo(*args, **kwargs):
    host = next(iter(args), kwargs.get("host"))
    pinned = _PINNED_HOSTS.get(host)
    if pinned is None:
        return _ORIGINAL_GETADDRINFO(*args, **kwargs)
    kwargs.pop("host", None)
    return _ORIGINAL_GETADDRINFO(pinned, *args[1:], **kwargs)


# Pin at the socket layer, so both our probe and OSMnx reach the machine we chose.
socket.gethostbyname = _pinned_gethostbyname
socket.getaddrinfo = _pinned_getaddrinfo

# Small enough that a healthy server answers instantly and an unhealthy one is not
# burdened: count the ways in a 50 m box.
_PROBE = "[out:json][timeout:25];way(55.5925,12.9995,55.5930,13.0005);out count;"

NO_ENDPOINT_MSG = (
    "No Overpass endpoint answered. It is a free, shared, volunteer-funded service and "
    "it throttles heavy clients: wait a few minutes and re-run. Tiles already downloaded "
    "are cached and will not be re-fetched, so a re-run resumes rather than restarts. "
    "You can also add another mirror to OVERPASS_CANDIDATES."
)


def _serves(base, timeout=30):
    """Does this endpoint answer a real query with real JSON?"""
    try:
        reply = requests.post(f"{base}/interpreter", data={"data": _PROBE},
                              timeout=timeout,
                              headers={"User-Agent": ox.settings.http_user_agent})
    except Exception as exc:                       # noqa: BLE001
        return False, type(exc).__name__
    if reply.status_code != 200:
        return False, f"HTTP {reply.status_code}"
    try:
        reply.json()
    except ValueError:
        return False, "non-JSON reply"
    return True, "OK"


def select_overpass(candidates=OVERPASS_CANDIDATES, timeout=30, quiet=False,
                    required=True):
    """Point OSMnx at the first candidate that answers a real query.

    With required=False, returns None instead of raising - so the configuration cell
    still finishes and the rest of the notebook stays inspectable while Overpass is
    down."""
    report = []
    for base in candidates:
        hostname = base.split("//", 1)[1].split("/", 1)[0]
        try:
            addresses = sorted({info[4][0] for info in _ORIGINAL_GETADDRINFO(
                hostname, 443, proto=socket.IPPROTO_TCP)})
        except OSError as exc:
            report.append(f"  {hostname:<26} DNS failed ({type(exc).__name__})")
            continue

        for ip in addresses:
            _PINNED_HOSTS[hostname] = ip
            ok, why = _serves(base, timeout)
            report.append(f"  {hostname:<26} {ip:<16} {why}")
            if ok:
                ox.settings.overpass_url = base
                if not quiet:
                    print("\n".join(report))
                    print(f"  -> using {base} via {ip}")
                return base
            del _PINNED_HOSTS[hostname]

    if not quiet:
        print("\n".join(report))
    if required:
        raise RuntimeError(NO_ENDPOINT_MSG)
    return None


# A server can die mid-download, so a retry re-selects the endpoint rather than hammering
# the one that just failed. Only transport-level and server-error failures are retried:
# "this query matched nothing" is a real answer and must surface at once.
_RETRYABLE = ("ConnectionError", "Timeout", "ChunkedEncodingError", "JSONDecodeError",
              "ResponseStatusCodeError", "RemoteDisconnected", "ProtocolError")


def with_retry(fn, *args, attempts=8, wait=20, **kwargs):
    import time
    for attempt in range(attempts):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:                   # noqa: BLE001
            name = type(exc).__name__
            if attempt == attempts - 1 or not any(r in name for r in _RETRYABLE):
                raise
            print(f"  attempt {attempt + 1} failed ({name}: {str(exc)[:70]})")
            base = select_overpass(quiet=True, required=False)
            print(f"  {'re-selected ' + base if base else 'no endpoint up'}; "
                  f"retrying in {wait}s")
            time.sleep(wait)


print("Overpass endpoints:")
if select_overpass(required=False) is None:
    print("\n  WARNING - no endpoint is answering right now. The cells below that do not\n"
          "  need a download will still run; the download cells will retry and, if\n"
          "  Overpass is still down, fail with:\n"
          f"    {NO_ENDPOINT_MSG}")

# Split the network download into tiles. The default lets a 16 km^2 box go out as one
# ~5 MB request, which is the first thing an overloaded Overpass server refuses. At
# 4 km^2 the routing box goes out as four small requests instead - and because OSMnx
# caches each one separately, a failure costs one tile rather than the whole download.
ox.settings.max_query_area_size = 4_000_000

# If Overpass is rate-limiting (HTTP 429) or refusing, point at a mirror instead.
# ox.settings.overpass_url = "https://overpass.kumi.systems/api"

_extra_tags = [
    "cycleway", "cycleway:left", "cycleway:right", "cycleway:both",
    "bicycle", "segregated", "foot", "surface", "maxspeed", "oneway:bicycle",
]
ox.settings.useful_tags_way = list(
    dict.fromkeys(list(ox.settings.useful_tags_way) + _extra_tags)
)

print(f"AOI            {AOI_SIDE_M:.0f} m square centred on {CENTER_LAT}, {CENTER_LON}")
print(f"facility box   AOI + {FACILITY_SEARCH_BUFFER_M:.0f} m")
print(f"routing box    AOI + {ROUTING_BUFFER_M:.0f} m")
print(f"impedance      {CYCLING_SPEED_KMH:.0f} km/h, "
      f"comfort weights {'ON' if USE_COMFORT_WEIGHTS else 'OFF'} {STRESS_FACTOR}")

## 2. Design tokens — White Arkitekter palette

The palette comes from `IN/Color Pallette White Arkitekter.png`. Chrome — surface, ink,
hairline grid, the recessive street ground — uses brand hexes exactly. The *data* marks
hold the brand **hues** but are deepened, for a reason the next cell measures rather
than asserts.

Two colour jobs appear here, and they are kept apart:

- **Categorical** — which facility a catchment belongs to. Identity, no order.
- **Ordinal / sequential** — minutes to the nearest facility. Magnitude, one hue,
  light → dark.

Catchments are a *partition of the plane*, so only **adjacent** catchments ever touch.
Rather than rely on that, the five categorical slots are validated on the harder
**all-pairs** test, and catchments are then assigned colours by graph colouring (§9), so
two catchments that share a border can never take the same slot.

In [ ]:
# ---------------------------------------------------------------------------
# White Arkitekter digital palette (IN/Color Pallette White Arkitekter.png)
# ---------------------------------------------------------------------------
W_MINT        = "#89D0C8"
W_TEAL        = "#389BAA"
W_WHITE       = "#ffffff"
W_BLACK       = "#000000"
W_GREY        = "#777777"
W_DENIM       = "#A1B2BF"
W_DARK_DENIM  = "#6A88A0"
W_PINK        = "#F49AC1"
W_PURPLE      = "#B586A5"
W_MISTY_GREEN = "#C5D5CB"
W_FORREST     = "#6C8C78"
W_SAND        = "#B0A89B"
W_SLATE       = "#7A9193"

# --- chrome: brand-exact -----------------------------------------------------
SURFACE   = W_WHITE
INK       = W_BLACK          # titles, facility labels
INK_2     = W_GREY           # subtitles, legend text
INK_MUTED = "#9e9e9e"        # axis ticks - brand Grey, lightened
GRID      = W_MISTY_GREEN    # hairline grid and AOI frame
GROUND    = W_DENIM          # the street network, when it is context rather than data

# --- categorical: catchment identity -----------------------------------------
# Five slots, one per brand hue, deepened to clear the chroma floor. Hue drift from
# the brand hex is at most 5 degrees, so each slot still reads as its brand colour.
#   slot  brand source        hue    role
#   0     Sand    #B0A89B      80    ochre
#   1     Forrest #6C8C78     156    green
#   2     Teal    #389BAA     209    teal   (brand Teal, nudged just over the floor)
#   3     Dark Denim #6A88A0  248    blue
#   4     Purple  #B586A5     345    plum
CAT = ["#b5820c", "#006439", "#2ea0b0", "#0066ab", "#964576"]
CAT_SOURCE = ["Sand", "Forrest", "Teal", "Dark Denim", "Purple"]

# --- ordinal: minutes to the nearest facility --------------------------------
# One hue (brand Teal, 209 deg), six monotone lightness steps, light = quick.
SEQ_TEAL = ["#29bfd3", "#0caabd", "#0095a7", "#008090", "#006c7a", "#005965"]

# --- annotation --------------------------------------------------------------
C_FACILITY = W_BLACK         # facility markers and their labels
C_UNSERVED = W_GREY          # demand beyond the cutoff
C_FLOW     = "#005965"       # loaded corridors (darkest step of the teal ramp)

print("categorical:", " ".join(CAT))
print("sequential :", " ".join(SEQ_TEAL))

### Colour validation — computed, not claimed

The palette rules used here are the ones a chart can be *checked* against: an OKLCH
lightness band and chroma floor per slot, and pairwise separation in OKLab (ΔE, ×100)
under normal vision plus protanopia and deuteranopia simulated with the
Machado–Oliveira–Fernandes 2009 model at severity 1.0.

Thresholds: categorical ΔE ≥ 8 simulated and ≥ 15 normal; chroma ≥ 0.10; lightness
0.43–0.77; ≥ 3:1 contrast on white. For the ordinal ramp: monotone lightness, adjacent
ΔL ≥ 0.06, light end ≥ 2:1.

The cell below runs those checks on the tokens above and prints the result. **This is
also why the brand hexes could not be used raw for the data marks:** run the same
function on them and the chroma floor fails — brand Teal sits at chroma 0.092, Denim at
0.027 — which is a statement about thin map lines, not about the palette. It is built
for large print and web fills, where those values are right.

In [ ]:
# --- sRGB <-> OKLab, and the Machado 2009 CVD matrices at severity 1.0 -------
_M_PROTAN = np.array([[0.152286, 1.052583, -0.204868],
                      [0.114503, 0.786281,  0.099216],
                      [-0.003882, -0.048116, 1.051998]])
_M_DEUTAN = np.array([[0.367322, 0.860646, -0.227968],
                      [0.280085, 0.672501,  0.047413],
                      [-0.011820, 0.042940, 0.968881]])


def _srgb(hex_colour):
    h = hex_colour.lstrip("#")
    return np.array([int(h[i:i + 2], 16) / 255 for i in (0, 2, 4)])


def _to_linear(c):
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)


def _oklab(linear_rgb):
    m = np.array([[0.4122214708, 0.5363325363, 0.0514459929],
                  [0.2119034982, 0.6806995451, 0.1073969566],
                  [0.0883024619, 0.2817188376, 0.6299787005]])
    n = np.cbrt(m @ linear_rgb)
    return np.array([[0.2104542553,  0.7936177850, -0.0040720468],
                     [1.9779984951, -2.4285922050,  0.4505937099],
                     [0.0259040371,  0.7827717662, -0.8086757660]]) @ n


def oklch(hex_colour):
    L, a, b = _oklab(_to_linear(_srgb(hex_colour)))
    return L, float(np.hypot(a, b)), float(np.degrees(np.arctan2(b, a)) % 360)


def delta_e(c1, c2, kind=None):
    """Euclidean distance in OKLab x100, optionally under simulated CVD."""
    mats = {"protan": _M_PROTAN, "deutan": _M_DEUTAN}
    out = []
    for colour in (c1, c2):
        lin = _to_linear(_srgb(colour))
        if kind:
            lin = np.clip(mats[kind] @ lin, 0, 1)
        out.append(_oklab(lin))
    return 100 * float(np.linalg.norm(out[0] - out[1]))


def contrast(c1, c2):
    lum = [float(np.dot([0.2126, 0.7152, 0.0722], _to_linear(_srgb(c)))) for c in (c1, c2)]
    hi, lo = max(lum), min(lum)
    return (hi + 0.05) / (lo + 0.05)


def check_categorical(palette, surface=SURFACE):
    rows, verdicts = [], []
    for colour in palette:
        L, C, H = oklch(colour)
        rows.append({"hex": colour, "L": round(L, 3), "C": round(C, 3),
                     "H": round(H), "contrast": round(contrast(colour, surface), 2)})
    frame = pd.DataFrame(rows)
    verdicts.append(("lightness band 0.43-0.77", frame["L"].between(0.43, 0.77).all(),
                     f"{frame['L'].min():.3f}-{frame['L'].max():.3f}"))
    verdicts.append(("chroma floor >= 0.10", (frame["C"] >= 0.10).all(),
                     f"min {frame['C'].min():.3f}"))
    verdicts.append(("contrast >= 3:1", (frame["contrast"] >= 3.0).all(),
                     f"min {frame['contrast'].min():.2f}:1"))
    for kind, floor in [(None, 15.0), ("protan", 8.0), ("deutan", 8.0)]:
        pairs = [(a, b, delta_e(a, b, kind))
                 for i, a in enumerate(palette) for b in palette[i + 1:]]
        worst = min(pairs, key=lambda t: t[2])
        verdicts.append((f"all-pairs dE >= {floor:.0f} ({kind or 'normal'})",
                         worst[2] >= floor, f"worst {worst[0]}<->{worst[1]} dE {worst[2]:.1f}"))
    return frame, verdicts


def check_ordinal(ramp, surface=SURFACE):
    ls = [oklch(c)[0] for c in ramp]
    hues = [oklch(c)[2] for c in ramp]
    gaps = [ls[i] - ls[i + 1] for i in range(len(ls) - 1)]
    return [
        ("lightness monotone light->dark", all(g > 0 for g in gaps), f"{ls[0]:.2f}->{ls[-1]:.2f}"),
        ("adjacent dL >= 0.06", min(gaps) >= 0.06, f"min {min(gaps):.3f}"),
        ("light end >= 2:1", contrast(ramp[0], surface) >= 2.0,
         f"{contrast(ramp[0], surface):.2f}:1"),
        ("single hue (spread <= 15 deg)", max(hues) - min(hues) <= 15,
         f"{max(hues) - min(hues):.0f} deg"),
    ]


def report(title, verdicts):
    print(title)
    print("-" * len(title))
    for name, passed, detail in verdicts:
        print(f"  {'PASS' if passed else 'FAIL'}  {name:<34} {detail}")
    print()


cat_frame, cat_verdicts = check_categorical(CAT)
display(cat_frame)
report("Categorical - 5 catchment slots, all-pairs", cat_verdicts)
report("Ordinal - 6-step teal ramp", check_ordinal(SEQ_TEAL))
report("Raw brand hexes, same test (this is why the marks are deepened)",
       check_categorical([W_SAND, W_FORREST, W_TEAL, W_DARK_DENIM, W_PURPLE])[1])

## 3. Study area

Three nested squares, all built in SWEREF99 TM so the AOI is exactly 1.00 km², then
reprojected to WGS84 for the Overpass queries.

In [ ]:
_center_m = gpd.GeoSeries([Point(CENTER_LON, CENTER_LAT)], crs=CRS_WGS84).to_crs(CRS_METRIC)


def square(side_m):
    """cap_style=3 buffers a point into a square: side = 2 * distance."""
    geom_m = _center_m.buffer(side_m / 2.0, cap_style=3).iloc[0]
    geom_wgs = gpd.GeoSeries([geom_m], crs=CRS_METRIC).to_crs(CRS_WGS84).iloc[0]
    return geom_m, geom_wgs


AOI_M, AOI_WGS84 = square(AOI_SIDE_M)
FAC_M, FAC_WGS84 = square(AOI_SIDE_M + 2 * FACILITY_SEARCH_BUFFER_M)
NET_M, NET_WGS84 = square(AOI_SIDE_M + 2 * ROUTING_BUFFER_M)

aoi_gdf = gpd.GeoDataFrame({"name": ["Triangeln AOI"]}, geometry=[AOI_M], crs=CRS_METRIC)

for label, geom in [("AOI", AOI_M), ("facility search", FAC_M), ("routing", NET_M)]:
    print(f"{label:<16} {math.sqrt(geom.area):>6.0f} m square   {geom.area / 1e6:>6.2f} km^2")

w, s, e, n = AOI_WGS84.bounds
print(f"\nAOI bbox (WGS84)   W {w:.5f}  S {s:.5f}  E {e:.5f}  N {n:.5f}")
print(f"AOI bbox (SWEREF)  {tuple(round(v, 1) for v in AOI_M.bounds)}")

## 4. The cycling network and its impedance

`network_type="bike"` gives the ways a bicycle may legally use, as a directed graph — so
one-way restrictions that apply to bikes are respected. The graph is **not** collapsed to
undirected here (unlike the fragmentation notebook, where the question was symmetric):
a route to a facility and back can legitimately differ.

In [ ]:
G_raw = with_retry(
    graph_from_polygon,
    NET_WGS84,
    network_type="bike",
    simplify=True,
    retain_all=True,        # keep every piece; dropping islands would hide poor access
    truncate_by_edge=True,
)
G = project_graph(G_raw, to_crs=CRS_METRIC)

print(f"nodes {G.number_of_nodes():,}   edges {G.number_of_edges():,}")
print(f"type {type(G).__name__}   directed {G.is_directed()}   crs {G.graph['crs']}")

### Traffic stress per edge

The same three-class scheme as the companion fragmentation notebook: **protected**
(physically separated from motor traffic), **painted lane** (marked, but in the
carriageway), **mixed** (nothing). OSM records provision both as a way of its own
(`highway=cycleway`) and as a tag on a road (`cycleway=track|lane` and its
`:left`/`:right`/`:both` variants), so the classifier reads both.

OSMnx returns a *list* wherever it merged parallel OSM ways into one simplified edge, so
every tag lookup normalises to a set first.

In [ ]:
PROTECTED, LANE, MIXED = "protected", "lane", "mixed"

CYCLEWAY_KEYS = ["cycleway", "cycleway:left", "cycleway:right", "cycleway:both"]
PROTECTED_CW_VALUES = {"track", "opposite_track", "sidepath"}
LANE_CW_VALUES = {
    "lane", "opposite_lane", "buffered_lane", "shared_lane",
    "share_busway", "opposite_share_busway", "crossing",
}
PATHY_HIGHWAYS = {"path", "footway", "pedestrian", "track", "bridleway"}
BIKE_ALLOWED = {"designated", "yes"}


def tagset(value):
    """Normalise an OSM tag value to a lowercase set; handles None, NaN and lists."""
    if value is None:
        return set()
    if isinstance(value, float) and math.isnan(value):
        return set()
    if isinstance(value, (list, tuple, set)):
        out = set()
        for item in value:
            out |= tagset(item)
        return out
    text = str(value).strip().lower()
    return {text} if text and text != "nan" else set()


def classify_edge(data):
    highway = tagset(data.get("highway"))
    bicycle = tagset(data.get("bicycle"))
    cycleway = set()
    for key in CYCLEWAY_KEYS:
        cycleway |= tagset(data.get(key))

    if "cycleway" in highway:
        return PROTECTED
    if cycleway & PROTECTED_CW_VALUES:
        return PROTECTED
    if INCLUDE_SHARED_PATHS and (highway & PATHY_HIGHWAYS) and (bicycle & BIKE_ALLOWED):
        return PROTECTED
    if cycleway & LANE_CW_VALUES:
        return LANE
    return MIXED


# metres per minute at the configured cycling speed
MPM = CYCLING_SPEED_KMH * 1000.0 / 60.0

for u, v, k, data in G.edges(keys=True, data=True):
    infra = classify_edge(data)
    length = float(data.get("length", 0.0))
    t_min = length / MPM
    data["infra"] = infra
    data["t_min"] = t_min
    data["t_perceived"] = t_min * STRESS_FACTOR[infra]

# The impedance every routing call in this notebook uses.
IMPEDANCE = "t_perceived" if USE_COMFORT_WEIGHTS else "t_min"

edges = graph_to_gdfs(G, nodes=False)
nodes = graph_to_gdfs(G, edges=False)

mix = (edges.groupby("infra")["length"].sum() / 1000).rename("km").to_frame()
mix["share_%"] = (100 * mix["km"] / mix["km"].sum()).round(1)
mix["stress_factor"] = [STRESS_FACTOR[i] for i in mix.index]
print(f"routing impedance: {IMPEDANCE}   ({CYCLING_SPEED_KMH:.0f} km/h = {MPM:.0f} m/min)\n")
display(mix.round(2))

## 5. Facilities — from OSM features to routable sites

OSM does not hand you a clean facility list. Three problems have to be fixed before any
allocation is meaningful:

1. **Mixed geometry.** A school is a polygon in one place and a node in another.
2. **Sub-features.** `Skånes Universitetssjukhus` is one campus, but its departments
   (*Ögonmottagning*, *Lungavdelning*, *Endokrinologimottagning* …) are separately mapped
   nodes *inside* it. Counted naively, one hospital becomes five, and the allocation
   invents catchment boundaries inside a single building complex.
3. **Duplicates.** The same school is occasionally mapped twice.

The resolver below fixes all three, and is deliberately asymmetric about it: a *point*
inside a site polygon is absorbed as a sub-feature, and duplicates are merged by name or
by proximity — but two *polygons* are never merged on distance, because neighbouring
schools routinely share a fence and merging them would erase a real facility.

Each surviving site is then snapped to the **nearest network node to the site geometry**
— not to its centroid. For a large campus those differ by a lot, and the boundary is
where a rider actually arrives.

In [ ]:
def fetch_facilities(tags, polygon):
    """Query OSM for one facility tier; return a GeoDataFrame in the metric CRS."""
    frames = []
    for key, values in tags.items():
        try:
            got = with_retry(features_from_polygon, polygon, {key: values})
        except Exception as exc:                       # noqa: BLE001
            # "Nothing matched" is a normal answer and must be distinguished from
            # "the download failed" - otherwise a dead network reads as a district
            # with no schools in it, which is a wrong answer rather than an error.
            if type(exc).__name__ != "InsufficientResponseError":
                raise
            print(f"  {key}={values}: no features matched")
            continue
        if len(got):
            frames.append(got)
    if not frames:
        return gpd.GeoDataFrame(
            {"name": [], "osm_kind": []}, geometry=[], crs=CRS_METRIC)

    raw = pd.concat(frames)
    raw = raw[~raw.index.duplicated(keep="first")]
    raw = raw.to_crs(CRS_METRIC)
    raw = raw[raw.geometry.notna() & ~raw.geometry.is_empty]

    keep = [c for c in ("name", "amenity", "healthcare", "operator", "isced:level")
            if c in raw.columns]
    out = raw[keep].copy()
    out["geometry"] = raw.geometry
    out["osm_kind"] = [g.geom_type for g in raw.geometry]
    return gpd.GeoDataFrame(out, geometry="geometry", crs=CRS_METRIC)


def resolve_sites(raw, absorb_m=SITE_ABSORB_M, dedupe_m=SITE_DEDUPE_M):
    """Collapse OSM features into distinct facility sites.

    Three rules, deliberately conservative about what counts as *one* site:

    1. Every polygon feature is a site. Two polygons are **never** merged on distance —
       neighbouring schools often share a fence, and merging them would erase a real
       facility.
    2. A point feature within `absorb_m` of a site polygon is a sub-feature of it (a
       hospital department inside its campus) and is absorbed.
    3. Remaining points within `dedupe_m` of each other, or any two clusters sharing a
       name, are the same site mapped twice.
    """
    if not len(raw):
        return gpd.GeoDataFrame(
            {"site_id": [], "name": [], "n_features": [], "area_m2": []},
            geometry=[], crs=CRS_METRIC)

    names = (raw["name"].astype("object") if "name" in raw.columns
             else pd.Series(None, index=raw.index, dtype="object"))
    is_poly = raw.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    polys, pts = raw[is_poly], raw[~is_poly]

    # Rule 1 + 2: polygons seed the clusters; points fall into the polygon they sit in.
    seeds = list(polys.geometry)
    members = [[idx] for idx in polys.index]
    loose = []
    for idx, geom in pts.geometry.items():
        hit = next((i for i, seed in enumerate(seeds) if seed.distance(geom) <= absorb_m),
                   None)
        if hit is None:
            loose.append((idx, geom))
        else:
            members[hit].append(idx)

    # Rule 3: cluster whatever is left.
    for idx, geom in loose:
        hit = next((i for i in range(len(polys), len(seeds))
                    if seeds[i].distance(geom) <= dedupe_m), None)
        if hit is None:
            seeds.append(geom)
            members.append([idx])
        else:
            seeds[hit] = unary_union([seeds[hit], geom])
            members[hit].append(idx)

    # Rule 3, name arm: the same facility mapped twice, wherever it sits.
    def cluster_name(idxs):
        got = [str(v) for v in names.loc[idxs].dropna()]
        return got[0] if got else None

    by_name = {}
    keep = []
    for i, idxs in enumerate(members):
        key = cluster_name(idxs)
        if key is not None and key in by_name:
            j = by_name[key]
            seeds[j] = unary_union([seeds[j], seeds[i]])
            members[j] = members[j] + idxs
            continue
        if key is not None:
            by_name[key] = i
        keep.append(i)

    rows = []
    for i in keep:
        idxs, geom = members[i], seeds[i]
        # Name the site after its largest member - the campus, not a department.
        sub = raw.loc[idxs]
        order = sub.geometry.area.fillna(0.0).sort_values(ascending=False).index
        named = [str(v) for v in names.loc[order].dropna()]
        rows.append({
            "site_id": len(rows),
            "name": named[0] if named else f"unnamed site {len(rows)}",
            "n_features": len(idxs),
            "area_m2": float(geom.area),
            "geometry": geom,
        })
    sites = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_METRIC)
    return sites.sort_values("area_m2", ascending=False).reset_index(drop=True)


def snap_to_network(gdf, node_gdf, how="geometry"):
    """Nearest graph node to each row. `how="geometry"` measures to the whole geometry
    (right for a campus), `how="centroid"` to its centroid."""
    probe = gdf.copy()
    if how == "centroid":
        probe["geometry"] = probe.geometry.centroid
    joined = gpd.sjoin_nearest(
        probe[["geometry"]], node_gdf[["geometry"]],
        how="left", distance_col="snap_m",
    )
    # sjoin_nearest emits one row per tied nearest node; keep one, and reindex so the
    # result is aligned to the input by index rather than by position.
    joined = joined[~joined.index.duplicated(keep="first")].reindex(gdf.index)

    out = gdf.copy()
    out["node"] = joined["index_right"].to_numpy()
    out["snap_m"] = joined["snap_m"].to_numpy()
    unsnapped = int(out["node"].isna().sum())
    if unsnapped:
        print(f"  dropped {unsnapped} feature(s) with no reachable network node")
        out = out[out["node"].notna()]
    out["node"] = out["node"].astype("int64")
    return out


facilities = {}
for kind, spec in FACILITY_SPECS.items():
    print(f"{spec['plural']}:")
    raw = fetch_facilities(spec["tags"], FAC_WGS84)
    sites = resolve_sites(raw)
    sites = snap_to_network(sites, nodes)
    sites["kind"] = kind
    sites["in_aoi"] = sites.geometry.intersects(AOI_M)
    sites["site_id"] = range(len(sites))
    facilities[kind] = sites
    absorbed = int(sites["n_features"].sum() - len(sites)) if len(sites) else 0
    print(f"  {len(raw):>3} OSM features -> {len(sites):>3} sites "
          f"({absorbed} sub-features absorbed), {int(sites['in_aoi'].sum())} inside the AOI")
    print(f"  snap distance to network: median {sites['snap_m'].median():.0f} m, "
          f"max {sites['snap_m'].max():.0f} m")

In [ ]:
for kind, sites in facilities.items():
    print(f"\n{FACILITY_SPECS[kind]['label']} sites ({len(sites)}):")
    view = sites[["site_id", "name", "n_features", "area_m2", "snap_m", "in_aoi"]].copy()
    view["area_m2"] = view["area_m2"].round(0)
    view["snap_m"] = view["snap_m"].round(0)
    display(view.head(50))

> **A finding, before any routing.** The hospital tier resolves to a very small number of
> sites, because central Malmö has essentially **one** hospital: the Skånes
> Universitetssjukhus campus. A nearest-*hospital* question in this window therefore has
> only one possible answer, and the closest-facility *allocation* for hospitals is
> degenerate — there is nothing to allocate between. What remains meaningful for that
> tier is the **access-time** half of the analysis, which §8 and §10 report in full.
>
> The pipeline below runs both tiers identically and flags the degenerate case rather
> than hiding it. If you want a hospital tier with real competition between sites, the
> Swedish first point of care is the *vårdcentral*: add `"clinic"` and `"doctors"` to the
> `amenity` list in `FACILITY_SPECS["hospital"]` and re-run. That is a different question
> — primary-care access, not hospital access — so it is not the default here.

## 6. Demand — residential buildings inside the AOI

Demand is placed on residential building footprints inside the AOI, weighted by a
**floor-area proxy**: footprint × storeys, using `building:levels` where OSM has it and
`DEFAULT_LEVELS` where it does not. Floor area is a better stand-in for people than
footprint alone in a district of perimeter blocks, where a tall block and a low one
occupy similar ground.

Each building is snapped to the nearest network node from its centroid. The snap
distance is reported: it is the part of the journey this model does not route.

In [ ]:
buildings_raw = with_retry(
    features_from_polygon, AOI_WGS84, {"building": True}).to_crs(CRS_METRIC)
buildings_raw = buildings_raw[
    buildings_raw.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

# Clip to the AOI so a block straddling the boundary contributes only its inside part.
buildings_raw["geometry"] = buildings_raw.geometry.intersection(AOI_M)
buildings_raw = buildings_raw[~buildings_raw.geometry.is_empty]

btag = buildings_raw["building"].map(lambda v: (sorted(tagset(v)) or [""])[0])
area = buildings_raw.geometry.area

keep = (~btag.isin(NON_RESIDENTIAL)) & (area >= MIN_BUILDING_AREA_M2)
demand = buildings_raw[keep].copy()
demand["btype"] = btag[keep]
demand["footprint_m2"] = demand.geometry.area


def levels_of(value):
    for item in tagset(value):
        try:
            return max(1.0, float(item))
        except ValueError:
            continue
    return DEFAULT_LEVELS


demand["levels"] = (demand["building:levels"].map(levels_of)
                    if "building:levels" in demand.columns else DEFAULT_LEVELS)
demand["levels_tagged"] = (demand["building:levels"].notna()
                           if "building:levels" in demand.columns else False)
demand["floor_area_m2"] = demand["footprint_m2"] * demand["levels"]

WEIGHT_COL = {"floor_area": "floor_area_m2", "footprint": "footprint_m2",
              "equal": "equal"}[DEMAND_WEIGHT]
if WEIGHT_COL == "equal":
    demand["equal"] = 1.0

demand = demand[["btype", "footprint_m2", "levels", "levels_tagged",
                 "floor_area_m2", WEIGHT_COL, "geometry"]].copy()
demand = demand.loc[:, ~demand.columns.duplicated()]
demand = snap_to_network(demand, nodes, how="centroid").reset_index(drop=True)
demand["w"] = demand[WEIGHT_COL]

far = demand["snap_m"] > SNAP_WARN_M
print(f"buildings in AOI          {len(buildings_raw):,}")
print(f"residential demand points {len(demand):,}")
print(f"weight                    {DEMAND_WEIGHT} ({WEIGHT_COL})")
print(f"total weight              {demand['w'].sum():,.0f}")
print(f"storeys tagged in OSM     {demand['levels_tagged'].sum():,} of {len(demand):,}"
      f"  (rest assumed {DEFAULT_LEVELS:.0f})")
print(f"snap to network           median {demand['snap_m'].median():.0f} m, "
      f"p95 {demand['snap_m'].quantile(0.95):.0f} m, "
      f"{int(far.sum())} beyond {SNAP_WARN_M:.0f} m")
display(demand["btype"].value_counts().head(10).rename("buildings").to_frame())

## 7. The nearest-facility solve

The naive way to answer "which facility is nearest" is one Dijkstra per demand point.
With *D* demand points and *F* facilities that is *D* searches. The standard trick is to
invert it: **one multi-source Dijkstra per facility tier**, seeded at every facility at
once, run on the **reversed** graph.

Reversing matters. The graph is directed, and the cost a rider pays is
cost(*home* → *facility*). A search outward from a facility on the *forward* graph would
compute cost(*facility* → *home*), which is a different number wherever a one-way is
involved. Running that same outward search on the reversed graph computes exactly
cost(*home* → *facility*) for every home at once.

Each settled node inherits three things from the node that relaxed it:

- `cost` — minutes to the nearest facility;
- `label` — *which* facility that is (this is what makes the allocation in §9 free);
- `nxt` — the next hop toward it **in the forward graph**, so a real route can be
  rebuilt for any home without a second search.

One pass, `O(E log V)`, and it answers both the nearest-facility and the
closest-facility-allocation question.

In [ ]:
# copy=False returns a read-only *view*: the reversed graph shares node and edge data
# with G, so nothing is duplicated in memory and edge keys are preserved. That key
# correspondence is what lets a reversed-graph search hand back forward-graph edges.
G_rev = G.reverse(copy=False)


def solve_nearest(rev_graph, seed_nodes, weight, cutoff=math.inf):
    """Multi-source Dijkstra on the reversed graph.

    seed_nodes : {node -> facility id}. Several facilities may share a node; the first
                 one listed wins that node, which is the correct tie-break (cost 0 to both).
    Returns (cost, label, nxt) keyed by node. `nxt[v] = (u, k)` is the forward edge
    v -> u that steps toward the nearest facility.
    """
    cost = {node: 0.0 for node in seed_nodes}
    label = dict(seed_nodes)
    nxt = {}
    heap = [(0.0, node) for node in seed_nodes]
    heapq.heapify(heap)

    while heap:
        d, u = heapq.heappop(heap)
        if d > cost.get(u, math.inf):
            continue  # stale heap entry
        # Out-edges of u in the reversed graph are in-edges of u in the forward graph,
        # so the forward edge is v -> u.
        for _, v, k, data in rev_graph.edges(u, keys=True, data=True):
            nd = d + float(data[weight])
            if nd > cutoff or nd >= cost.get(v, math.inf):
                continue
            cost[v] = nd
            label[v] = label[u]
            nxt[v] = (u, k)
            heapq.heappush(heap, (nd, v))

    return cost, label, nxt


def seeds_for(sites):
    """{node -> site_id}, keeping the first site when two snap to the same node."""
    out = {}
    for row in sites.itertuples():
        out.setdefault(row.node, row.site_id)
    return out


def edge_geometry(graph, u, v, k):
    data = graph.edges[u, v, k]
    geom = data.get("geometry")
    if geom is not None:
        return geom
    return LineString([(graph.nodes[u]["x"], graph.nodes[u]["y"]),
                       (graph.nodes[v]["x"], graph.nodes[v]["y"])])


def route_edges(nxt, start):
    """The forward edges (u, v, key) a rider actually uses, home -> facility."""
    out, node, seen = [], start, {start}
    while node in nxt:
        nxt_node, key = nxt[node]
        out.append((node, nxt_node, key))
        if nxt_node in seen:        # impossible on a shortest-path tree; cheap guard
            break
        seen.add(nxt_node)
        node = nxt_node
    return out


def real_costs(graph, sol):
    """True ride minutes and metres to the nearest facility, for every reached node.

    The `nxt` map is a shortest-path tree, so this is one bottom-up pass over it rather
    than a walk per node: each node's cost is its successor's cost plus one edge.
    """
    minutes, metres = {}, {}
    nxt = sol["nxt"]
    for start in sol["cost"]:
        stack, node = [], start
        while node not in minutes and node in nxt:
            stack.append(node)
            node = nxt[node][0]
        minutes.setdefault(node, 0.0)      # a seed node, or already resolved
        metres.setdefault(node, 0.0)
        while stack:
            v = stack.pop()
            u, k = nxt[v]
            data = graph.edges[v, u, k]    # forward edge v -> u
            minutes[v] = minutes[u] + float(data["t_min"])
            metres[v] = metres[u] + float(data["length"])
    return minutes, metres


solutions = {}
for kind, sites in facilities.items():
    if not len(sites):
        print(f"{kind}: no sites found - skipped")
        continue
    seeds = seeds_for(sites)
    cost, label, nxt = solve_nearest(G_rev, seeds, IMPEDANCE)
    sol = {"cost": cost, "label": label, "nxt": nxt, "seeds": seeds}
    # Real ride time and distance along the chosen (possibly comfort-optimal) route.
    sol["real_min"], sol["real_m"] = real_costs(G, sol)
    solutions[kind] = sol
    reached = len(cost)
    print(f"{FACILITY_SPECS[kind]['label']:<9} {len(sites):>3} sites, "
          f"{len(seeds):>3} distinct network nodes -> "
          f"{reached:,} of {G.number_of_nodes():,} nodes reached "
          f"({100 * reached / G.number_of_nodes():.1f}%)")

### Attach the answer to each home

For each demand point: the nearest facility, the perceived cost the router minimised,
and the **real** ride time and distance along that same route — which is what a person
experiences, and is not the same number when comfort weights are on.

In [ ]:
for kind, sol in solutions.items():
    cost, label = sol["cost"], sol["label"]
    real_min, real_m = sol["real_min"], sol["real_m"]

    demand[f"{kind}_site"] = [label.get(n) for n in demand["node"]]
    demand[f"{kind}_perceived_min"] = [cost.get(n, np.nan) for n in demand["node"]]
    demand[f"{kind}_min"] = [real_min.get(n, np.nan) for n in demand["node"]]
    demand[f"{kind}_m"] = [real_m.get(n, np.nan) for n in demand["node"]]

    minutes = demand[f"{kind}_min"]
    unreached = int(demand[f"{kind}_site"].isna().sum())
    print(f"{FACILITY_SPECS[kind]['label']:<9} "
          f"median {minutes.median():.1f} min, "
          f"p90 {minutes.quantile(0.90):.1f} min, "
          f"max {minutes.max():.1f} min"
          + (f"   [{unreached} homes unreachable]" if unreached else ""))

display(demand[[c for c in demand.columns
                if c.endswith(("_site", "_min", "_m")) and not c.startswith("snap")]]
        .head(8).round(2))

## 8. Access: how long is the ride?

The headline table. `p90` is the number that matters for a service standard — it is the
ride the *worst-served tenth* of the district faces, and a mean hides it.

In [ ]:
def weighted_quantile(values, weights, q):
    values, weights = np.asarray(values, float), np.asarray(weights, float)
    ok = np.isfinite(values) & np.isfinite(weights)
    values, weights = values[ok], weights[ok]
    if not len(values):
        return np.nan
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cum = np.cumsum(weights) - 0.5 * weights
    return float(np.interp(q * weights.sum(), cum, values))


access_rows = []
for kind in solutions:
    col, w = f"{kind}_min", demand["w"]
    ok = demand[col].notna()
    target = COVERAGE_TARGET_MIN.get(kind, 10.0)
    access_rows.append({
        "facility": FACILITY_SPECS[kind]["label"],
        "sites reachable": int(demand.loc[ok, f"{kind}_site"].nunique()),
        "median min": weighted_quantile(demand[col], w, 0.50),
        "mean min": float(np.average(demand.loc[ok, col], weights=w[ok])),
        "p90 min": weighted_quantile(demand[col], w, 0.90),
        "max min": float(demand[col].max()),
        "median m": weighted_quantile(demand[f"{kind}_m"], w, 0.50),
        "% within target": 100 * w[ok & (demand[col] <= target)].sum() / w.sum(),
        "target min": target,
    })

access = pd.DataFrame(access_rows).set_index("facility")
print("Cycling access from residential floor area inside the 1 km AOI")
display(access.round(2))

### Coverage curve

The share of residential floor area that can reach its nearest facility within *t*
minutes by bike. Two series, so the chart carries a legend; the target thresholds from
`COVERAGE_TARGET_MIN` are marked directly on the curves rather than in a separate key.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))
grid_t = np.linspace(0, 16, 321)
total_w = demand["w"].sum()

for slot, kind in enumerate(solutions):
    col = demand[f"{kind}_min"]
    share = [100 * demand.loc[col <= t, "w"].sum() / total_w for t in grid_t]
    colour = CAT[2] if kind == "school" else CAT[0]
    ax.plot(grid_t, share, color=colour, linewidth=2.0, zorder=3,
            label=FACILITY_SPECS[kind]["label"])

    target = COVERAGE_TARGET_MIN.get(kind, 10.0)
    hit = 100 * demand.loc[col <= target, "w"].sum() / total_w
    ax.plot([target], [hit], marker="o", markersize=8, color=colour,
            markeredgecolor=SURFACE, markeredgewidth=2, zorder=5)
    ax.annotate(f"{hit:.0f}% within {target:.0f} min",
                xy=(target, hit), xytext=(8, -14), textcoords="offset points",
                fontsize=9.5, color=INK, fontweight="bold", zorder=6)

ax.set_xlim(0, 16)
ax.set_ylim(0, 102)
ax.set_xlabel("cycling minutes to the nearest facility", fontsize=10, color=INK_2)
ax.set_ylabel("% of residential floor area", fontsize=10, color=INK_2)
ax.set_title("How much of the district reaches a facility within t minutes",
             fontsize=14, fontweight="bold", color=INK, loc="left", pad=12)
ax.text(0.0, 1.02, f"Triangeln 1 km AOI - by bicycle at {CYCLING_SPEED_KMH:.0f} km/h"
        + (", comfort-weighted routing" if USE_COMFORT_WEIGHTS else ""),
        transform=ax.transAxes, fontsize=10, color=INK_2, va="bottom")
ax.yaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="both", length=0, labelsize=9.5, colors=INK_MUTED)
ax.legend(frameon=False, fontsize=10, labelcolor=INK_2, loc="lower right")

fig.tight_layout()
fig.savefig(OUT_DIR / "triangeln_bike_coverage_curve.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Closest-facility allocation

The `label` map from §7 already assigns every network node to its nearest facility, so
the allocation is read straight off it — no second solve.

An **edge** belongs to a catchment when both of its endpoints do. Where the two endpoints
carry different labels, the edge straddles a catchment boundary: that is the network
equivalent of a watershed divide, the point along the street where the nearest facility
changes.

Catchments are drawn as a ribbon around their allocated streets rather than as a filled
tessellation, because that is what the analysis actually knows. A network allocation says
nothing about the interior of a block; painting solid polygons would claim precision the
method does not have.

In [ ]:
# Cartography window: the AOI plus a margin, so catchments are not cut at the frame.
DISPLAY_MARGIN_M = 300.0
DISPLAY_M = AOI_M.buffer(DISPLAY_MARGIN_M, cap_style=3)
_dx0, _dy0, _dx1, _dy1 = DISPLAY_M.bounds
edges_disp = edges.cx[_dx0:_dx1, _dy0:_dy1]


def allocate(sol, edge_gdf):
    """Split edges into (allocated, boundary) and return the catchment adjacency graph."""
    label = sol["label"]
    site_of, boundary = [], []
    adjacency = defaultdict(set)
    for (u, v, k) in edge_gdf.index:
        a, b = label.get(u), label.get(v)
        if a is not None and a == b:
            site_of.append(a)
            boundary.append(False)
        else:
            site_of.append(a if b is None else (b if a is None else None))
            boundary.append(a is not None and b is not None and a != b)
            if a is not None and b is not None and a != b:
                adjacency[a].add(b)
                adjacency[b].add(a)
    out = edge_gdf.copy()
    out["site"] = site_of
    out["is_boundary"] = boundary
    return out, adjacency


def colour_catchments(site_ids, adjacency, n_slots=len(CAT)):
    """Welsh-Powell greedy colouring: two catchments sharing a border never share a slot."""
    order = sorted(site_ids, key=lambda s: (-len(adjacency.get(s, ())), s))
    slot = {}
    for site in order:
        taken = {slot[n] for n in adjacency.get(site, ()) if n in slot}
        free = [c for c in range(n_slots) if c not in taken]
        slot[site] = free[0] if free else min(
            range(n_slots), key=lambda c: sum(slot.get(n) == c for n in adjacency.get(site, ())))
    clashes = sum(1 for a, nbrs in adjacency.items() for b in nbrs
                  if a < b and slot.get(a) == slot.get(b))
    return slot, clashes


def catchment_polygons(alloc, slot, band=CATCHMENT_BAND_M, clip=None):
    rows = []
    for site, part in alloc[alloc["site"].notna()].groupby("site"):
        # A ribbon around the allocated streets, lightly eroded so the shape reads as a
        # corridor rather than a claim about the interior of every block.
        geom = unary_union(list(part.geometry)).buffer(band).buffer(-band * 0.25)
        if clip is not None:
            geom = geom.intersection(clip)
        if geom.is_empty:
            continue
        rows.append({"site": int(site), "slot": slot.get(site, 0),
                     "network_m": float(part["length"].sum()),
                     "area_m2": float(geom.area), "geometry": geom})
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_METRIC)


allocations = {}
for kind, sol in solutions.items():
    alloc, adjacency = allocate(sol, edges_disp)
    present = sorted({int(s) for s in alloc["site"].dropna().unique()})
    slot, clashes = colour_catchments(present, adjacency)
    catch = catchment_polygons(alloc, slot, clip=DISPLAY_M)
    allocations[kind] = {"edges": alloc, "adjacency": adjacency,
                         "slot": slot, "catchments": catch}
    print(f"{FACILITY_SPECS[kind]['label']:<9} {len(present):>2} catchments in the window, "
          f"{int(alloc['is_boundary'].sum()):>4} boundary edges, "
          f"{len(set(slot.values()))} colour slots used, {clashes} adjacent clashes")

### Per-facility load

Which facility each home is allocated to, aggregated. `share %` is the fraction of the
AOI's residential floor area that this facility is the nearest one for — an
**exposure** measure, not an enrolment forecast: it says how much of the district would
come here if everyone simply went to their nearest.

In [ ]:
load_tables = {}
for kind, sol in solutions.items():
    sites = facilities[kind].set_index("site_id")
    col, w = f"{kind}_min", "w"
    rows = []
    for site, part in demand.dropna(subset=[f"{kind}_site"]).groupby(f"{kind}_site"):
        site = int(site)
        catch = allocations[kind]["catchments"]
        area = catch.loc[catch["site"] == site, "area_m2"]
        rows.append({
            "site": site,
            "name": sites.at[site, "name"],
            "buildings": len(part),
            "floor_area_m2": part[w].sum(),
            "median min": weighted_quantile(part[col], part[w], 0.50),
            "p90 min": weighted_quantile(part[col], part[w], 0.90),
            "catchment_ha": float(area.iloc[0]) / 1e4 if len(area) else np.nan,
            # Fall back to slot 0 for a site whose catchment lies outside the map window.
            "slot": allocations[kind]["slot"].get(site, 0),
        })
    table = pd.DataFrame(rows).sort_values("floor_area_m2", ascending=False)
    table["slot"] = table["slot"].fillna(0).astype(int)
    table["share_%"] = 100 * table["floor_area_m2"] / demand[w].sum()
    load_tables[kind] = table.reset_index(drop=True)

    print(f"\n{FACILITY_SPECS[kind]['label']} - allocation of AOI residential floor area")
    if len(table) == 1:
        print("  Degenerate allocation: one site takes the whole AOI "
              "(see the note in section 5).")
    display(table[["site", "name", "buildings", "floor_area_m2", "share_%",
                   "median min", "p90 min", "catchment_ha"]].round(1))

## 10. Maps

### 10.1 School catchments

The allocation, drawn as it would be on a plan: catchment ribbons over a recessive street
ground, boundary streets picked out as a dashed divide, and each school marked and
numbered. Catchment colours come from the graph colouring in §9, so two catchments that
share a border never share a colour — and the number on the marker, not the hue, is what
identifies a school.

In [ ]:
def draw_base(ax, window=DISPLAY_M):
    ground = edges_disp
    ground.plot(ax=ax, color=GROUND, linewidth=0.55, zorder=1)
    aoi_gdf.boundary.plot(ax=ax, color=INK, linewidth=1.1, linestyle=(0, (6, 3)), zorder=8)
    x0, y0, x1, y1 = window.bounds
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal")
    ax.set_axis_off()


def mark_facilities(ax, sites, ids, fontsize=8.0):
    shown = sites[sites["site_id"].isin(ids)]
    for row in shown.itertuples():
        pt = row.geometry.representative_point()
        if not DISPLAY_M.contains(pt):
            continue
        ax.annotate(str(row.site_id), xy=(pt.x, pt.y),
                    ha="center", va="center", fontsize=fontsize, fontweight="bold",
                    color=SURFACE, zorder=10,
                    bbox=dict(boxstyle="circle,pad=0.30", facecolor=C_FACILITY,
                              edgecolor=SURFACE, linewidth=1.4))


kind = "school"
if kind in allocations and len(load_tables[kind]) > 1:
    alloc = allocations[kind]["edges"]
    catch = allocations[kind]["catchments"]
    slot = allocations[kind]["slot"]
    table = load_tables[kind]

    fig, ax = plt.subplots(figsize=(11, 11))
    draw_base(ax)

    for row in catch.itertuples():
        gpd.GeoSeries([row.geometry], crs=CRS_METRIC).plot(
            ax=ax, color=CAT[row.slot], alpha=0.30, edgecolor="none", zorder=2)

    for site, part in alloc[alloc["site"].notna()].groupby("site"):
        part.plot(ax=ax, color=CAT[slot[int(site)]], linewidth=1.7, zorder=4)

    bnd = alloc[alloc["is_boundary"]]
    if len(bnd):
        bnd.plot(ax=ax, color=INK, linewidth=1.4, linestyle=(0, (1.5, 1.8)), zorder=6)

    mark_facilities(ax, facilities[kind], table["site"])

    top = table.head(6)
    handles = [Patch(facecolor=CAT[int(slot_i)], edgecolor="none",
                     label=f"{int(site_i)}  {str(name_i)[:26]} - {share_i:.0f}%")
               for site_i, name_i, slot_i, share_i
               in zip(top["site"], top["name"], top["slot"], top["share_%"])]
    handles += [
        Line2D([], [], color=INK, lw=1.4, ls=(0, (1.5, 1.8)), label="catchment boundary"),
        Line2D([], [], color=GROUND, lw=1.0, label="street, unallocated / outside"),
        Line2D([], [], color=INK, lw=1.1, ls=(0, (6, 3)), label="1 km AOI"),
    ]
    ax.legend(handles=handles, loc="lower left", frameon=False, fontsize=8.6,
              labelcolor=INK_2, handlelength=1.6, borderpad=0.8)

    ax.set_title("Which school is nearest by bike", fontsize=15, fontweight="bold",
                 color=INK, loc="left", pad=14)
    ax.text(0.0, 1.005,
            f"Closest-facility allocation of the cycling network - "
            f"{len(table)} schools capture the 1 km AOI"
            + (", comfort-weighted" if USE_COMFORT_WEIGHTS else ""),
            transform=ax.transAxes, fontsize=10, color=INK_2, va="bottom")

    fig.tight_layout()
    fig.savefig(OUT_DIR / "triangeln_school_catchments.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Allocation map needs at least two competing sites.")

### 10.2 Ride time to the nearest facility

The same ordinal teal ramp on both panels, so the two are directly comparable: one hue,
light = quick, six bands. Small multiples rather than one map with two ramps — a reader
should never have to hold two colour scales at once.

In [ ]:
BAND_EDGES = [0] + TIME_BANDS_MIN + [np.inf]
BAND_LABELS = ([f"under {TIME_BANDS_MIN[0]}"]
               + [f"{a}-{b}" for a, b in zip(TIME_BANDS_MIN, TIME_BANDS_MIN[1:])]
               + [f"over {TIME_BANDS_MIN[-1]}"])

kinds = list(solutions)
fig, axes = plt.subplots(1, len(kinds), figsize=(7.2 * len(kinds), 7.8))
axes = np.atleast_1d(axes)

for ax, kind in zip(axes, kinds):
    real_min = solutions[kind]["real_min"]
    # An edge takes the band of its upstream node: the time a rider is at when using it.
    band = [np.nan if u not in real_min else np.digitize(real_min[u], BAND_EDGES) - 1
            for (u, v, k) in edges_disp.index]
    layer = edges_disp.copy()
    layer["band"] = band

    draw_base(ax)
    for b in range(len(BAND_LABELS)):
        part = layer[layer["band"] == b]
        if len(part):
            part.plot(ax=ax, color=SEQ_TEAL[min(b, len(SEQ_TEAL) - 1)],
                      linewidth=1.5 + 0.12 * b, zorder=3 + b)
    mark_facilities(ax, facilities[kind], facilities[kind]["site_id"])

    target = COVERAGE_TARGET_MIN.get(kind, 10.0)
    hit = 100 * demand.loc[demand[f"{kind}_min"] <= target, "w"].sum() / demand["w"].sum()
    ax.set_title(FACILITY_SPECS[kind]["label"], fontsize=14, fontweight="bold",
                 color=INK, loc="left", pad=10)
    ax.text(0.0, 1.005,
            f"median {access.loc[FACILITY_SPECS[kind]['label'], 'median min']:.1f} min - "
            f"{hit:.0f}% of floor area within {target:.0f} min",
            transform=ax.transAxes, fontsize=10, color=INK_2, va="bottom")

handles = [Line2D([], [], color=SEQ_TEAL[i], lw=2.4 + 0.2 * i, label=f"{lab} min")
           for i, lab in enumerate(BAND_LABELS)]
handles += [Line2D([], [], marker="o", linestyle="none", markersize=8,
                   markerfacecolor=C_FACILITY, markeredgecolor=SURFACE, label="facility"),
            Line2D([], [], color=INK, lw=1.1, ls=(0, (6, 3)), label="1 km AOI")]
axes[0].legend(handles=handles, loc="lower left", frameon=False, fontsize=9,
               labelcolor=INK_2, handlelength=1.8, borderpad=0.8,
               title="cycling minutes to nearest", title_fontsize=9)

fig.suptitle("Cycling time to the nearest facility", fontsize=15, fontweight="bold",
             color=INK, x=0.005, ha="left", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "triangeln_bike_access_time.png", dpi=200, bbox_inches="tight")
plt.show()

### 10.3 Where the school run goes

The allocation says *which* school; this says *which streets*. Every home's route to its
nearest school is walked and its floor-area weight added to each edge it uses, so line
width is the load a street carries. These are the corridors where cycling provision has
the most riders per metre — and where a gap in the protected network is felt by the most
people.

In [ ]:
def load_network(sol, weight_col="w"):
    """Accumulate demand weight onto every forward edge a route actually uses.

    The edges come from the `nxt` tree, so this is the route the solver chose - not a
    re-derived one, which would silently pick a different parallel edge.
    """
    flow = defaultdict(float)
    for node, w in zip(demand["node"], demand[weight_col]):
        if node not in sol["cost"]:
            continue
        for edge in route_edges(sol["nxt"], node):
            flow[edge] += float(w)
    return flow


kind = "school"
flow = load_network(solutions[kind])
flow_gdf = gpd.GeoDataFrame(
    [{"u": u, "v": v, "key": k, "load": w,
      "infra": G.edges[u, v, k]["infra"], "geometry": edge_geometry(G, u, v, k)}
     for (u, v, k), w in flow.items() if w > 0],
    geometry="geometry", crs=CRS_METRIC,
)
flow_gdf = flow_gdf[flow_gdf.geometry.intersects(DISPLAY_M)]
flow_gdf["load_share"] = flow_gdf["load"] / demand["w"].sum()

if not len(flow_gdf):
    raise RuntimeError("No school trips were loaded onto the network - check the solve.")

# Four load bands, quantile-cut. One list drives both the drawing and the legend, so the
# two cannot drift apart.
_q = flow_gdf["load"].quantile([0.0, 0.5, 0.8, 0.95]).to_numpy()
LOAD_BANDS = [
    (_q[0], _q[1], 0.9, 0.45, "light"),
    (_q[1], _q[2], 1.8, 0.65, "moderate"),
    (_q[2], _q[3], 3.2, 0.85, "busy"),
    (_q[3], np.inf, 5.4, 1.00, "top 5% of links"),
]

fig, ax = plt.subplots(figsize=(11, 11))
draw_base(ax)

for lo, hi, lw, alpha, _label in LOAD_BANDS:
    part = flow_gdf[(flow_gdf["load"] >= lo) & (flow_gdf["load"] < hi)]
    if len(part):
        part.plot(ax=ax, color=C_FLOW, linewidth=lw, alpha=alpha, zorder=4)

mark_facilities(ax, facilities[kind], load_tables[kind]["site"])

peak = float(flow_gdf["load"].max())
handles = [Line2D([], [], color=C_FLOW, lw=lw, alpha=alpha, label=label)
           for _lo, _hi, lw, alpha, label in LOAD_BANDS]
handles += [Line2D([], [], marker="o", linestyle="none", markersize=8,
                   markerfacecolor=C_FACILITY, markeredgecolor=SURFACE, label="school"),
            Line2D([], [], color=GROUND, lw=1.0, label="street carrying no school trip"),
            Line2D([], [], color=INK, lw=1.1, ls=(0, (6, 3)), label="1 km AOI")]
ax.legend(handles=handles, loc="lower left", frameon=False, fontsize=9, labelcolor=INK_2,
          handlelength=1.8, borderpad=0.8, title="school-trip load", title_fontsize=9)

ax.set_title("The school run, loaded onto the cycling network", fontsize=15,
             fontweight="bold", color=INK, loc="left", pad=14)
ax.text(0.0, 1.005,
        "Every AOI home routed to its nearest school; width is floor area carried. "
        f"Busiest link carries {100 * peak / demand['w'].sum():.0f}% of the district.",
        transform=ax.transAxes, fontsize=10, color=INK_2, va="bottom")

fig.tight_layout()
fig.savefig(OUT_DIR / "triangeln_school_run_load.png", dpi=200, bbox_inches="tight")
plt.show()

### What kind of street carries the load?

The load per infrastructure class, against the share of network length each class
provides. A class carrying more load than length is doing more than its share of the work.

In [ ]:
by_infra = flow_gdf.assign(load_m=flow_gdf["load"] * flow_gdf.geometry.length)
summary = by_infra.groupby("infra")["load_m"].sum().rename("load x metres").to_frame()
summary["load_share_%"] = 100 * summary["load x metres"] / summary["load x metres"].sum()
net_km = edges_disp.groupby("infra")["length"].sum() / 1000
summary["network_share_%"] = 100 * net_km / net_km.sum()
summary["over/under"] = summary["load_share_%"] / summary["network_share_%"]
display(summary.drop(columns=["load x metres"]).round(2))

## 11. Load balance across schools

Bar colour is the map colour from §10.1, so chart and map read together; the school name
on the axis is what carries identity, and the value label sits on the bar. The dashed
line is an even split — where every school would sit if the district's floor area were
shared equally, which is the reference the eye needs to see imbalance.

In [ ]:
kind = "school"
table = load_tables[kind]
if len(table) > 1:
    top = table.head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 0.48 * len(top) + 2.4))

    labels = [f"{int(s)}  {str(n)[:32]}" for s, n in zip(top["site"], top["name"])]
    colours = [CAT[int(s)] for s in top["slot"]]
    shares = top["share_%"].to_numpy()
    homes = top["buildings"].to_numpy()
    medians = top["median min"].to_numpy()

    ax.barh(labels, shares, height=0.62, color=colours, zorder=3)

    even = 100.0 / len(table)
    ax.axvline(even, color=INK_2, linewidth=1.1, linestyle=(0, (4, 3)), zorder=4)
    ax.annotate(f"even split ({even:.0f}%)", xy=(even, len(top) - 0.45),
                xytext=(5, 0), textcoords="offset points",
                fontsize=9, color=INK_2, va="center")

    span = float(shares.max())
    for label, share, n_homes, med in zip(labels, shares, homes, medians):
        ax.text(share + span * 0.02, label, f"{share:.0f}%", va="center",
                fontsize=9.5, color=INK, fontweight="bold", zorder=5)
        ax.text(share + span * 0.11, label,
                f"{int(n_homes)} homes - median {med:.0f} min",
                va="center", fontsize=8.5, color=INK_MUTED, zorder=5)

    ax.set_xlim(0, span * 1.38)
    ax.set_xlabel("% of the AOI's residential floor area allocated to this school",
                  fontsize=10, color=INK_2)
    ax.set_title("Not an even split: which schools the district's homes fall to",
                 fontsize=14, fontweight="bold", color=INK, loc="left", pad=12)
    ax.text(0.0, 1.02, "Closest-facility allocation by bike, Triangeln 1 km AOI",
            transform=ax.transAxes, fontsize=10, color=INK_2, va="bottom")
    ax.xaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="both", length=0, labelsize=9.5, colors=INK_2)

    fig.tight_layout()
    fig.savefig(OUT_DIR / "triangeln_school_load_balance.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Load balance needs at least two competing sites.")

## 12. Does comfort change the answer?

Everything above routed on `t_perceived` — time inflated by traffic stress. Re-solving on
plain `t_min` gives the distance-optimal answer, and the difference is the part of this
analysis that a car-network or straight-line study cannot see:

- **Reassigned** — homes whose *nearest school changes* once mixed traffic is priced in.
  For these, the closest school and the school a rider would actually choose are not the
  same building.
- **Detour** — the extra real minutes a comfort-seeking rider accepts to stay off traffic.

In [ ]:
compare_rows = []
for kind, sites in facilities.items():
    if len(sites) < 2:
        compare_rows.append({"facility": FACILITY_SPECS[kind]["label"],
                             "reassigned %": np.nan, "mean detour min": np.nan,
                             "note": "single site - nothing to reassign"})
        continue

    seeds = seeds_for(sites)
    cost_t, label_t, nxt_t = solve_nearest(G_rev, seeds, "t_min")
    fast_min, _ = real_costs(G, {"cost": cost_t, "nxt": nxt_t})

    changed_w = same_w = 0.0
    detour = []
    for node, w, t_comfy in zip(demand["node"], demand["w"], demand[f"{kind}_min"]):
        if node not in cost_t or node not in solutions[kind]["cost"]:
            continue
        w = float(w)
        if label_t[node] != solutions[kind]["label"][node]:
            changed_w += w
        else:
            same_w += w
        detour.append((float(t_comfy) - fast_min[node], w))

    detour = np.array(detour)
    total = changed_w + same_w
    compare_rows.append({
        "facility": FACILITY_SPECS[kind]["label"],
        "reassigned %": 100 * changed_w / total if total else np.nan,
        "mean detour min": float(np.average(detour[:, 0], weights=detour[:, 1]))
        if len(detour) else np.nan,
        "p90 detour min": float(weighted_quantile(detour[:, 0], detour[:, 1], 0.90))
        if len(detour) else np.nan,
        "note": "",
    })

comparison = pd.DataFrame(compare_rows).set_index("facility")
print("Comfort-weighted routing vs. distance-optimal routing")
print(f"stress factors: {STRESS_FACTOR}\n")
display(comparison.round(2))

## 13. Export

In [ ]:
def flatten(gdf):
    """GPKG cannot hold list-valued fields; collapse merged OSM tags to a string."""
    out = gdf.copy()
    for col in out.columns:
        if col == out.geometry.name:
            continue
        if out[col].map(lambda v: isinstance(v, (list, tuple, set))).any():
            out[col] = out[col].map(lambda v: ", ".join(sorted(tagset(v))) or None)
        elif out[col].dtype == object:
            out[col] = out[col].map(lambda v: v if v is None or isinstance(v, (str, float, int, bool)) else str(v))
    return out


layers = {"aoi": aoi_gdf, "demand_buildings": flatten(demand)}
for kind in facilities:
    layers[f"{kind}_sites"] = flatten(facilities[kind])
    if kind in allocations:
        layers[f"{kind}_catchments"] = flatten(allocations[kind]["catchments"])
        alloc = allocations[kind]["edges"].reset_index()
        keep = [c for c in ("u", "v", "key", "site", "is_boundary", "infra",
                            "length", "t_min", "t_perceived", "name", "geometry")
                if c in alloc.columns]
        layers[f"{kind}_allocated_edges"] = flatten(alloc[keep])
layers["school_run_load"] = flatten(flow_gdf)

gpkg = OUT_DIR / "triangeln_bike_facility_access.gpkg"
try:
    for name, gdf in layers.items():
        gdf.to_file(gpkg, layer=name, driver="GPKG")
    print(f"wrote {gpkg}\n  layers: {', '.join(layers)}")
except Exception as exc:                                   # noqa: BLE001
    print(f"GeoPackage write failed ({exc}); falling back to GeoJSON")
    for name, gdf in layers.items():
        gdf.to_file(OUT_DIR / f"triangeln_{name}.geojson", driver="GeoJSON")

access.to_csv(OUT_DIR / "triangeln_bike_access_summary.csv")
comparison.to_csv(OUT_DIR / "triangeln_bike_comfort_comparison.csv")
for kind, table in load_tables.items():
    table.to_csv(OUT_DIR / f"triangeln_{kind}_allocation.csv", index=False)

print("\ntables:")
for path in sorted(OUT_DIR.glob("triangeln_*bike*.csv")) + sorted(OUT_DIR.glob("triangeln_*allocation.csv")):
    print(f"  {path.name}")

## Findings

1. **Schools are a real allocation; hospitals are not.** Central Malmö has one hospital
   campus, so every home in the square has the same nearest hospital and the
   closest-facility question collapses to a pure access-time question. The school tier is
   where the allocation carries information.
2. **The split between schools is uneven.** Catchments are shaped by the cycling network,
   not by distance — the barrier effect of a rail cut or a hard-to-cross arterial pushes
   homes to a school that is further away in a straight line.
3. **Comfort changes who goes where.** §12 measures the share of homes whose nearest
   school changes once mixed traffic is priced into the impedance. Any home in that share
   is one where a distance-based catchment map is describing a trip nobody takes.
4. **The load map localises the investment case.** A handful of links carry a
   disproportionate share of the school run; those are where a gap in protected provision
   costs the most rider-metres — and they are the natural place to join this analysis to
   the missing-link ranking in the companion notebook.

## Caveats

- **Floor area is not children.** Demand is weighted by a residential floor-area proxy
  (footprint × storeys, `building:levels` where tagged and `DEFAULT_LEVELS` elsewhere),
  which says nothing about household composition. For a real school-planning number,
  substitute SCB DeSO population by age band.
- **Nearest is not chosen.** Swedish school choice is not a catchment system. This is an
  *accessibility* model — who is near what — not an enrolment model.
- **Facility resolution is heuristic.** Sub-features are absorbed by geometry and name.
  It fixes the SUS-campus case cleanly, but two genuinely different schools sharing a site
  would be merged. Check `n_features` in the site tables before quoting a facility count.
- **Stress factors are assumptions.** `STRESS_FACTOR` sets how much a minute in mixed
  traffic costs. The ordering is well supported; the magnitudes are a modelling choice, and
  §12 is the sensitivity test for them.
- **The first and last 100 m are not routed.** Homes and facilities are snapped to the
  nearest network node; the reported snap distances are the part of the journey outside
  the model. Facilities are snapped to the *nearest point of the site*, so a campus is
  entered at its edge rather than its centre — but not necessarily at a real gate.
- **Edge effects are buffered, not eliminated.** Facilities are searched to
  `FACILITY_SEARCH_BUFFER_M` past the AOI and routed to `ROUTING_BUFFER_M`. A facility
  beyond that is invisible to the model. Widen both together, never one alone.
- **Signals and dismounts are free.** Edge impedance is length ÷ speed × stress. Junction
  delay, waiting at a crossing and dismount points are not modelled, so all times are
  optimistic and most optimistic on the arterial-crossing routes.

## Next steps

- Swap the floor-area weight for SCB DeSO population by age band, and split the school
  tier by `isced:level` so a primary-school catchment is not competing with a gymnasium.
- Add capacity: a nearest-facility solve ignores whether the facility can take the load.
  A capacitated allocation (transportation problem) would show where the nearest school
  is already full.
- Intersect the school-run load map with the protected-network gaps from the companion
  fragmentation notebook, and re-rank missing links by *riders served* rather than by
  metres of network rescued.
- Re-run with `amenity=clinic|doctors` folded into the hospital tier to get the
  primary-care picture, where there is genuine competition between sites.